In [0]:

import json

from pyspark.sql import functions as F
from pyspark.sql.window import Window

from common_utils.logging import get_logger
from common_utils.scdfunctions import scd_type_1


# --------------------------------------------------
# 1. Logger
# --------------------------------------------------

logger = get_logger("bronze_to_silver_facts")

logger.info("Starting Bronze to Silver Fact processing")


# --------------------------------------------------
# 2. Read JSON configuration
# --------------------------------------------------

config_path = "/Workspace/Users/shashankkotagiri22@gmail.com/Employee-Data-Platform/src/silver/config/config_facts.json"

with open(config_path, "r") as file:
    configs = json.load(file)


# --------------------------------------------------
# 3. Deduplication function
# --------------------------------------------------

def apply_deduplication(df, dedup_config):

    method = dedup_config["method"]

    if method == "latest_record":

        window_spec = (
            Window
            .partitionBy(*dedup_config["partition_columns"])
            .orderBy(F.col(dedup_config["order_column"]).desc())
        )

        df = (
            df.withColumn("row_num", F.row_number().over(window_spec))
              .filter(F.col("row_num") == 1)
              .drop("row_num")
        )

    elif method == "drop_duplicates":

        df = df.dropDuplicates(dedup_config["columns"])

    else:
        raise ValueError(f"Unknown deduplication method: {method}")

    return df


# --------------------------------------------------
# 4. Additional transformation function
# --------------------------------------------------

def apply_additional_transformation(df, transformation):

    if transformation == "calculate_shrinkage_duration":

        df = df.withColumn(
            "shrinkage_duration_minutes",
            (
                F.col("end_datetime").cast("long")
                - F.col("start_datetime").cast("long")
            ) / 60
        )

    return df


# --------------------------------------------------
# 5. Main Bronze to Silver function
# --------------------------------------------------

def process_fact(fact_name):

    logger.info(f"Starting processing for {fact_name}")

    # Get configuration
    fact_config = configs[fact_name]

    source_tables = fact_config["source_tables"]
    target_table = fact_config["target_table"]
    key_columns = fact_config["key_columns"]

    # Read and combine Bronze tables
    df = spark.table(source_tables[0])

    for table_name in source_tables[1:]:
        df = df.unionByName(spark.table(table_name))

    logger.info(f"Combined Bronze tables for {fact_name}")

    # Deduplication
    df = apply_deduplication(
        df,
        fact_config["deduplication"]
    )

    logger.info(f"Deduplication completed for {fact_name}")

    # Drop technical columns
    df = df.drop(*fact_config["drop_columns"])

    # Additional transformation
    df = apply_additional_transformation(
        df,
        fact_config["additional_transformation"]
    )

    logger.info(f"Transformations completed for {fact_name}")

    # Create Silver schema
    spark.sql("""
        CREATE SCHEMA IF NOT EXISTS employeedataplatform.silver
    """)

    # Create target table if it does not exist
    if not spark.catalog.tableExists(target_table):

        df.write.format("delta") \
            .mode("overwrite") \
            .saveAsTable(target_table)

        logger.info(f"Created Silver table: {target_table}")

    # Apply SCD Type 1
    scd_type_1(
        spark,
        df,
        target_table,
        key_columns
    )

    logger.info(f"Completed processing for {fact_name}")

    return df


# --------------------------------------------------
# 6. Run FactWork
# --------------------------------------------------

fact_work = process_fact("fact_work")

print("FactWork processing completed")
print("Total Silver records:", fact_work.count())
fact_shrinkage = process_fact("fact_shrinkage")
print("Total Silver records:", fact_shrinkage.count())

fact_shift = process_fact("fact_shift")
print("Total Silver records:", fact_shift.count())
